# Phase 2: YOLOv12 Single-Class Training Pipeline (Google Colab Kernel)

This notebook is designed to execute on a Google Colab GPU kernel while connected from a local Jupyter interface.
All data and outputs persist on Google Drive.

### **Instructions**
Run the setup and verification cells (Cells 1-6). Review the output carefully. Do not run the final 100-epoch training cell until approval is granted.

In [ ]:
# 1. Environment and GPU Audit
!pip install ultralytics pyyaml pandas opencv-python-headless matplotlib > /dev/null 2>&1

import sys
import torch
import ultralytics

print("--- ENVIRONMENT AUDIT ---")
print(f"Python Version: {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
print(f"Ultralytics Version: {ultralytics.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU VRAM (GB): {round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2)}")
else:
    raise SystemError("No CUDA GPU detected! Please ensure you are connected to a Colab GPU kernel.")

In [ ]:
# 2. Mount Google Drive
import os

PROJECT_ROOT = '/content/drive/MyDrive/sem_defect_project'
DATASET_ROOT = f"{PROJECT_ROOT}/dataset_yolo_single_class"
ZIP_PATH = f"{PROJECT_ROOT}/dataset_yolo_single_class.zip"
RUNS_DIR = f"{PROJECT_ROOT}/runs/detect"

# Check if Drive is already mounted (either previously or via Colab Files sidebar)
if os.path.exists('/content/drive/MyDrive'):
    print("✅ Google Drive is already mounted and accessible.")
else:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        print("✅ Google Drive successfully mounted.")
    except Exception as e:
        print("⚠️ Notice: Remote Jupyter kernel cannot open interactive Colab auth popup.")
        print("👉 To fix: In your Colab web browser tab, click the 'Files' icon (left panel) -> click 'Mount Drive'.")
        if not os.path.exists('/content/drive/MyDrive'):
            raise RuntimeError("Google Drive is not mounted yet. Please click 'Mount Drive' in the Colab browser UI.") from e

os.makedirs(RUNS_DIR, exist_ok=True)
print(f"📁 Project root: {PROJECT_ROOT}")
print(f"📁 Runs directory: {RUNS_DIR}")

In [ ]:
# 3. Extract Dataset if Not Exists
import zipfile
import shutil

if not os.path.exists(DATASET_ROOT):
    if not os.path.exists(ZIP_PATH):
        found_files = os.listdir(PROJECT_ROOT) if os.path.exists(PROJECT_ROOT) else []
        raise FileNotFoundError(
            f"Dataset ZIP not found at '{ZIP_PATH}'.\n"
            f"Files currently in '{PROJECT_ROOT}': {found_files}\n"
            "Please ensure 'dataset_yolo_single_class.zip' is placed inside 'My Drive/sem_defect_project/'."
        )
    
    print(f"Extracting {ZIP_PATH} to {PROJECT_ROOT}...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(PROJECT_ROOT)
    print("✅ Extraction complete.")
else:
    print(f"✅ Dataset already exists at {DATASET_ROOT}")

In [ ]:
# 4. Dataset Verification & Path Rewriting
import yaml
import glob

print("--- DATASET VERIFICATION ---")
splits = ['train', 'valid', 'test']
for split in splits:
    img_dir = f"{DATASET_ROOT}/{split}/images"
    lbl_dir = f"{DATASET_ROOT}/{split}/labels"
    
    if not os.path.exists(img_dir) or not os.path.exists(lbl_dir):
        raise FileNotFoundError(f"Missing structure for split '{split}'. Expected {img_dir} and {lbl_dir}.")
    
    imgs = glob.glob(f"{img_dir}/*.jpg")
    print(f"Split '{split}': Found {len(imgs)} images.")
    if len(imgs) == 0:
        raise ValueError(f"No images found in {img_dir}")

DATA_YAML = f"{DATASET_ROOT}/data.yaml"
if not os.path.exists(DATA_YAML):
    raise FileNotFoundError(f"{DATA_YAML} not found!")

# Ensure data.yaml points to absolute paths for Colab
with open(DATA_YAML, 'r') as f:
    data = yaml.safe_load(f)

data['train'] = f"{DATASET_ROOT}/train/images"
data['val'] = f"{DATASET_ROOT}/valid/images"
data['test'] = f"{DATASET_ROOT}/test/images"

with open(DATA_YAML, 'w') as f:
    yaml.dump(data, f)

print("data.yaml updated with absolute Colab paths and verified.")

In [ ]:
# 5. YOLOv12 Model Loading Test
from ultralytics import YOLO
import gc

print("Loading YOLO model...")
try:
    model = YOLO("yolov12n.pt")
    print("Successfully loaded yolov12n.pt")
except Exception as e:
    print(f"YOLOv12 not found. Falling back to YOLO11. Error: {e}")
    model = YOLO("yolo11n.pt")
    print("Successfully loaded yolo11n.pt")

print("Model load test passed.")
del model
gc.collect()
torch.cuda.empty_cache()

---
### **STOP HERE**
**Do NOT proceed to the full training cell below until you have reviewed the verification outputs above.**

In [14]:
# 6. Dry Run (2 Epochs) - Pre-Flight Training Verification
from ultralytics import YOLO

print("Starting 2-epoch pre-flight dry run to verify training loop, loss calculation, and Drive checkpointing...")
model = YOLO("yolo11n.pt")

dry_results = model.train(
    data=DATA_YAML,
    epochs=2,
    imgsz=512,
    batch=16,
    optimizer="AdamW",
    lr0=0.001,
    project=RUNS_DIR,
    name="EXP-01-YOLO11-SingleClass-DryRun",
    seed=42,
    device=0,
    exist_ok=True,
    save=True,
    plots=True
)
print("✅ Dry run passed! Weights and metrics successfully verified on Google Drive.")

Starting 2-epoch pre-flight dry run to verify training loop, loss calculation, and Drive checkpointing...
Ultralytics 8.4.150 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/sem_defect_project/dataset_yolo_single_class/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=2, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=

---
### **Full Training**
Run the cell below for the full 100-epoch baseline training run.

In [15]:
# 7. Full 100-Epoch Training (YOLO11 Single-Class Defect Baseline)
from ultralytics import YOLO

print(f"Starting full 100-epoch training. Checkpoints will be saved to: {RUNS_DIR}")
model = YOLO("yolo11n.pt")

results = model.train(
    data=DATA_YAML,
    epochs=100,
    imgsz=512,
    batch=16,
    patience=20,
    optimizer="AdamW",
    lr0=0.001,
    project=RUNS_DIR,
    name="EXP-02-YOLO11-SingleClass-Baseline",
    seed=42,
    device=0,
    exist_ok=True,
    save=True,
    save_period=-1,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=0.0,
    translate=0.1,
    scale=0.5,
    flipud=0.5,
    fliplr=0.5,
    mosaic=0.5,
    plots=True
)
print("✅ Full training complete! All weights, curves, and validation metrics saved to Google Drive.")

Starting full 100-epoch training. Checkpoints will be saved to: /content/drive/MyDrive/sem_defect_project/runs/detect
Ultralytics 8.4.150 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/sem_defect_project/dataset_yolo_single_class/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.0